# ADB Indonesia — legacy PDF-pattern recovery (87 unresolved records)

Tests additional **historical** ADB filename conventions, including the double-slash form that worked for `43251-025`. Does **not** claim to search the complete ADB catalogue. No PDFs are marked verified unless they pass a PDF-header check; project-title matching must still be audited. The already verified `43251-025` RRP is excluded. Run cells in order.

In [ ]:
import requests, csv, time, io, zipfile, pathlib, hashlib
from google.colab import files
ROOT=pathlib.Path("/content/ADB_Legacy_Recovery"); ROOT.mkdir(exist_ok=True)
OUT=ROOT/"pdfs"; OUT.mkdir(exist_ok=True)
ROWS=[{'project_id': '38354-013', 'document_type': 'PCR', 'project_title': 'State Accountability Revitalization Project'}, {'project_id': '50297-001', 'document_type': 'PCR', 'project_title': 'Leveraging Private Infrastructure Investment Project'}, {'project_id': '50395-006', 'document_type': 'PCR', 'project_title': 'Advanced Knowledge and Skills for Sustainable Growth Project'}, {'project_id': '51114-001', 'document_type': 'PCR', 'project_title': 'â€” Electricity Grid Development Program (Phase 2)'}, {'project_id': '51157-001', 'document_type': 'PCR', 'project_title': 'Flood Management in North Java Project'}, {'project_id': '52282-001', 'document_type': 'PCR', 'project_title': 'Geothermal Power Generation Project'}, {'project_id': '52316-001', 'document_type': 'PCR', 'project_title': 'Emergency Assistance for Rehabilitation and Reconstruction'}, {'project_id': '52332-001', 'document_type': 'PCR', 'project_title': 'Higher Education for Technology and Innovation Project'}, {'project_id': '53211-003', 'document_type': 'PCR', 'project_title': 'Competitiveness, Industrial Modernization, and Trade Acceleration Program, Subprogram 3'}, {'project_id': '53220-001', 'document_type': 'PCR', 'project_title': 'Sustainable and Reliable Energy Access Program - Western and Central Java'}, {'project_id': '54152-001', 'document_type': 'PCR', 'project_title': 'Green Finance Facility (Phase 1)'}, {'project_id': '54224-001', 'document_type': 'PCR', 'project_title': 'Supporting Essential Health Actions and Transformation Program'}, {'project_id': '54224-002', 'document_type': 'PCR', 'project_title': 'Primary Healthcare and Public Health Laboratories Upgrading and Strengthening Project'}, {'project_id': '54256-001', 'document_type': 'PCR', 'project_title': 'Horticulture Development in Dryland Areas Sector Project'}, {'project_id': '54428-001', 'document_type': 'PCR', 'project_title': 'Citywide Inclusive Sanitation Project'}, {'project_id': '55020-001', 'document_type': 'PCR', 'project_title': 'Infrastructure Improvement for Shrimp Aquaculture Project'}, {'project_id': '55063-001', 'document_type': 'PCR', 'project_title': 'Promoting Research and Innovation through Modern and Efficient Science and Technology Parks Project'}, {'project_id': '56257-001', 'document_type': 'PCR', 'project_title': 'Domestic Resource Mobilization Program, Subprogram 1'}, {'project_id': '57074-001', 'document_type': 'RRP', 'project_title': 'Integrated Fishing Ports and International Fish Markets - Phase 2 Preparation Project'}, {'project_id': '57074-001', 'document_type': 'PCR', 'project_title': 'Integrated Fishing Ports and International Fish Markets - Phase 2 Preparation Project'}, {'project_id': '52282-003', 'document_type': 'PCR', 'project_title': 'Geothermal Power Generation Project (Second Additional Financing)'}, {'project_id': '54461-003', 'document_type': 'PCR', 'project_title': 'Boosting Productivity Through Human Capital Development Program (Subprogram 3)'}, {'project_id': '56017-001', 'document_type': 'PCR', 'project_title': 'Trans South-South Java Road Project'}, {'project_id': '56140-001', 'document_type': 'PCR', 'project_title': 'Transition Program (Phase 1)'}, {'project_id': '32507-013', 'document_type': 'RRP', 'project_title': 'Capital Market Development Program Cluster (Subprogram I) (formerly Financial Governance and Social Security Reform Program II)'}, {'project_id': '32507-013', 'document_type': 'PCR', 'project_title': 'Capital Market Development Program Cluster (Subprogram I) (formerly Financial Governance and Social Security Reform Program II)'}, {'project_id': '32507-023', 'document_type': 'RRP', 'project_title': 'Capital Market Development Program Cluster - Subprogram 2'}, {'project_id': '32507-023', 'document_type': 'PCR', 'project_title': 'Capital Market Development Program Cluster - Subprogram 2'}, {'project_id': '33409-013', 'document_type': 'RRP', 'project_title': 'Vocational Education Strengthening Project'}, {'project_id': '34100-013', 'document_type': 'RRP', 'project_title': 'Renewable Energy Development'}, {'project_id': '34152-013', 'document_type': 'RRP', 'project_title': 'Community Water Services and Health'}, {'project_id': '34152-013', 'document_type': 'PCR', 'project_title': 'Community Water Services and Health'}, {'project_id': '34159-013', 'document_type': 'RRP', 'project_title': 'Road Rehabilitation II'}, {'project_id': '34159-013', 'document_type': 'PCR', 'project_title': 'Road Rehabilitation II'}, {'project_id': '35139-013', 'document_type': 'RRP', 'project_title': 'Power Transmission Improvement Sector'}, {'project_id': '35182-043', 'document_type': 'PCR', 'project_title': 'Flood Management in Selected River Basins Sector Project'}, {'project_id': '35183-013', 'document_type': 'RRP', 'project_title': 'Sustainable Aquaculture Development for Food Security and Poverty Reduction'}, {'project_id': '36541-013', 'document_type': 'RRP', 'project_title': 'Local Government Finance and Governance Reform Program'}, {'project_id': '36541-013', 'document_type': 'PCR', 'project_title': 'Local Government Finance and Governance Reform Program'}, {'project_id': '36541-023', 'document_type': 'RRP', 'project_title': 'Local Government Finance and Governance Reform (Project Loan)'}, {'project_id': '36541-023', 'document_type': 'PCR', 'project_title': 'Local Government Finance and Governance Reform (Project Loan)'}, {'project_id': '37049-013', 'document_type': 'RRP', 'project_title': 'Multitranche Financing Facility - Integrated Citarum Water Resources Management Investment Program'}, {'project_id': '37049-023', 'document_type': 'RRP', 'project_title': 'Integrated Citarum Water Resources Management Investment Program - Project 1'}, {'project_id': '37475-013', 'document_type': 'RRP', 'project_title': 'Madrasah Education Development'}, {'project_id': '37475-013', 'document_type': 'PCR', 'project_title': 'Madrasah Education Development'}, {'project_id': '38117-013', 'document_type': 'RRP', 'project_title': 'Nutrition Improvement through Community Empowerment'}, {'project_id': '38264-013', 'document_type': 'RRP', 'project_title': 'Second Local Government Finance and Governance Reform Program - Subprogram 1'}, {'project_id': '38385-013', 'document_type': 'RRP', 'project_title': 'Rural Infrastructure Support to the PNPM Mandiri Project II'}, {'project_id': '38479-034', 'document_type': 'RRP', 'project_title': 'Regional Roads Development Project'}, {'project_id': '39071-013', 'document_type': 'RRP', 'project_title': 'Metropolitan Sanitation Management and Health Project'}, {'project_id': '39597-013', 'document_type': 'RRP', 'project_title': 'Rural Infrastructure Support Project'}, {'project_id': '39597-013', 'document_type': 'PCR', 'project_title': 'Rural Infrastructure Support Project'}, {'project_id': '40003-013', 'document_type': 'RRP', 'project_title': 'Poverty Reduction and Millennium Development Goals Acceleration Program'}, {'project_id': '40009-013', 'document_type': 'RRP', 'project_title': 'Infrastructure Project Development Facility'}, {'project_id': '40009-023', 'document_type': 'RRP', 'project_title': 'Infrastructure Reform Sector Development Program (Subprogram 1)'}, {'project_id': '40009-033', 'document_type': 'RRP', 'project_title': 'Infrastructure Reform Sector Development Program (Subprogram 2)'}, {'project_id': '40009-043', 'document_type': 'RRP', 'project_title': 'Infrastructure Reform Sector Development Program - Subprogram 3'}, {'project_id': '40061-013', 'document_type': 'RRP', 'project_title': 'Java-Bali Electricity Distribution Performance Improvement Project'}, {'project_id': '40247-013', 'document_type': 'RRP', 'project_title': 'Rural Infrastructure Support to PNPM Mandiri Project'}, {'project_id': '40247-013', 'document_type': 'PCR', 'project_title': 'Rural Infrastructure Support to PNPM Mandiri Project'}, {'project_id': '40272-013', 'document_type': 'RRP', 'project_title': 'Third Development Policy Support Program'}, {'project_id': '40272-013', 'document_type': 'PCR', 'project_title': 'Third Development Policy Support Program'}, {'project_id': '40273-013', 'document_type': 'RRP', 'project_title': 'Fourth Development Policy Support Program'}, {'project_id': '40273-013', 'document_type': 'PCR', 'project_title': 'Fourth Development Policy Support Program'}, {'project_id': '42099-013', 'document_type': 'RRP', 'project_title': 'Polytechnics Education Development Project'}, {'project_id': '42109-013', 'document_type': 'RRP', 'project_title': 'Company Project'}, {'project_id': '43009-013', 'document_type': 'RRP', 'project_title': 'Public Expenditure Support Facility Program'}, {'project_id': '43009-013', 'document_type': 'PCR', 'project_title': 'Public Expenditure Support Facility Program'}, {'project_id': '43092-013', 'document_type': 'RRP', 'project_title': 'Fifth Development Policy Support Program'}, {'project_id': '43092-013', 'document_type': 'PCR', 'project_title': 'Fifth Development Policy Support Program'}, {'project_id': '43211-013', 'document_type': 'RRP', 'project_title': 'Sixth Development Policy Support Program'}, {'project_id': '43211-013', 'document_type': 'PCR', 'project_title': 'Sixth Development Policy Support Program'}, {'project_id': '43251-025', 'document_type': 'PCR', 'project_title': 'Metropolitan Sanitation Management Investment Project'}, {'project_id': '43255-013', 'document_type': 'RRP', 'project_title': 'Urban Sanitation and Rural Infrastructure Support to PNPM Mandiri Project'}, {'project_id': '44252-013', 'document_type': 'RRP', 'project_title': 'Financial Market Development and Integration Program'}, {'project_id': '46093-004', 'document_type': 'RRP', 'project_title': 'Inclusive Growth through Improved Connectivity Program, Subprogram 2'}, {'project_id': '46239-001', 'document_type': 'RRP', 'project_title': 'Precautionary Financing Facility'}, {'project_id': '52218-001', 'document_type': 'PCR', 'project_title': 'Promoting Innovative Financial Inclusion Program (Subprogram 1)'}, {'project_id': '52218-002', 'document_type': 'PCR', 'project_title': 'Promoting Innovative Financial Inclusion Program (Subprogram 2)'}, {'project_id': '53211-001', 'document_type': 'PCR', 'project_title': 'Competitiveness, Industrial Modernization, and Trade Acceleration Program, Subprogram 1'}, {'project_id': '53211-002', 'document_type': 'PCR', 'project_title': 'Competitiveness, Industrial Modernization, and Trade Acceleration Program, Subprogram 2'}, {'project_id': '54461-001', 'document_type': 'PCR', 'project_title': 'Boosting Productivity Through Human Capital Development Program (Subprogram 1)'}, {'project_id': '54461-002', 'document_type': 'PCR', 'project_title': 'Boosting Productivity Through Human Capital Development Program, Subprogram 2'}, {'project_id': '55235-001', 'document_type': 'PCR', 'project_title': 'State-Owned Enterprisesâ€™ Reform Program, Subprogram 1'}, {'project_id': '56368-001', 'document_type': 'PCR', 'project_title': 'Affordable and Sustainable Energy Transition Program, Subprogram 1'}, {'project_id': '57018-001', 'document_type': 'PCR', 'project_title': 'Reducing Marine Debris Program, Subprogram 1'}, {'project_id': '58165-001', 'document_type': 'PCR', 'project_title': 'Promoting Innovative Financial Inclusion Program, Subprogram 3'}]
print("Rows:",len(ROWS),"RRP:",sum(r["document_type"]=="RRP" for r in ROWS),"PCR:",sum(r["document_type"]=="PCR" for r in ROWS))

## Run RRPs first
Change `TYPES` to `{"PCR"}` later only if the RRP test yields useful results. Each candidate is checked conservatively. If ADB returns access restrictions, the script records that and stops rather than trying to bypass restrictions.

In [ ]:
TYPES={"RRP"}
s=requests.Session(); s.headers.update({"User-Agent":"Mozilla/5.0 (compatible; academic-document-audit/1.0)"})
results=[]
for i,r in enumerate([x for x in ROWS if x["document_type"] in TYPES],1):
    pid=r["project_id"]; typ=r["document_type"].lower(); parent=pid.split("-")[0]
    # Includes historical double slash, single slash, and parent-folder variants.
    paths=[f"/sites/default/files/project-documents//{pid}-{typ}.pdf",f"/sites/default/files/project-documents/{pid}-{typ}.pdf",f"/sites/default/files/project-documents/{parent}/{pid}-{typ}.pdf",f"/sites/default/files/project-documents/{parent}/{pid}-{typ}-en.pdf"]
    status="NOT_FOUND_BY_PATTERN"; found=""; errors=[]; restricted=False
    for path in paths:
        url="https://www.adb.org"+path
        try:
            resp=s.get(url,timeout=22,allow_redirects=True)
            if resp.status_code in (401,403,429):
                status="ACCESS_RESTRICTED"; errors.append(f"{resp.status_code}: {url}"); restricted=True; break
            if resp.status_code==200 and resp.content[:5]==b"%PDF-":
                name=f"{pid}-{typ}.pdf"; (OUT/name).write_bytes(resp.content)
                status="DOWNLOADED"; found=url; break
            errors.append(f"{resp.status_code}: {url}")
        except requests.RequestException as e: errors.append(f"{type(e).__name__}: {url}")
        time.sleep(0.5)
    results.append({**r,"status":status,"confirmed_pdf_url":found,"attempt_log":" | ".join(errors)})
    with (ROOT/"legacy_audit.csv").open("w",newline="",encoding="utf-8") as f:
        writer=csv.DictWriter(f,fieldnames=list(results[0]));writer.writeheader();writer.writerows(results)
    print(f"[{i}] {pid} {typ.upper()}: {status}",flush=True)
    if restricted:
        print("ADB access restriction encountered; stopping requests.");break
print("Completed:",len(results),"New PDFs:",sum(x["status"]=="DOWNLOADED" for x in results))

In [ ]:
archive=ROOT/"ADB_Indonesia_Legacy_Recovery.zip"
with zipfile.ZipFile(archive,"w",zipfile.ZIP_DEFLATED) as z:
    for p in OUT.glob("*.pdf"):z.write(p,p.name)
    for p in ROOT.glob("*.csv"):z.write(p,p.name)
print("Archive:",archive,"size:",round(archive.stat().st_size/1024/1024,2),"MB")
files.download(str(archive))